In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x : '%.4f' % x)
import sys
sys.path.append("..")

# Paquetes de visualización
import seaborn as sns 
import matplotlib.pyplot as plt 
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Importaciones de unidades de soporte
from SRC import sp_metrics

In [2]:
aviation_df = pd.read_csv('../data/joint_dataset_operationalperformancemetrics.csv')

In [4]:
# Filtering completed flights

completed_flights = aviation_df[aviation_df["CANCELLED"] == 0].copy()

In [10]:
# Major KPIs table

total_flights = len(aviation_df)

otp = (completed_flights["ARRIVAL_DELAY"] <= 15).mean() * 100
delay_rate = 100 - otp
severe_delay_rate = (completed_flights["ARRIVAL_DELAY"] > 60).mean() * 100
cancel_rate = aviation_df["CANCELLED"].mean() * 100
avg_delay = completed_flights["ARRIVAL_DELAY"].mean()

kpi_table = pd.DataFrame({
    "Metric": [
        "Total Flights",
        "On-Time Performance (%)",
        "Delay Rate (%)",
        "Severe Delay Rate (%)",
        "Cancellation Rate (%)",
        "Average Arrival Delay (min)"
    ],
    "Value": [
        total_flights,
        round(otp,2),
        round(delay_rate,2),
        round(severe_delay_rate,2),
        round(cancel_rate,2),
        round(avg_delay,2)
    ]
})

kpi_table.to_csv('../data/joint_dataset_dashboard_kpis.csv', index=False)

In [11]:
# Delay distribution

bins = [-1000,0,15,30,60,120,10000]

labels = [
    "Early (<0)",
    "On Time (0-15)",
    "15-30",
    "30-60",
    "60-120",
    ">120"
]

completed_flights["DELAY_BUCKET"] = pd.cut(
    completed_flights["ARRIVAL_DELAY"],
    bins=bins,
    labels=labels
)

delay_distribution = completed_flights["DELAY_BUCKET"].value_counts(normalize=True) * 100

delay_distribution = delay_distribution.reset_index()

delay_distribution.columns = ["Delay Category","Percentage"]

delay_distribution = delay_distribution.round(2)

delay_distribution.to_csv('../data/joint_dataset_delay_distribution_dashboard.csv', index=False)

In [12]:
# Root cause contribution

delay_causes = [
    "AIR_SYSTEM_DELAY",
    "WEATHER_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "SECURITY_DELAY"
]

delay_cause_sum = completed_flights[delay_causes].sum()

delay_cause_percentage = (delay_cause_sum / delay_cause_sum.sum()) * 100

delay_cause_percentage = delay_cause_percentage.reset_index()

delay_cause_percentage.columns = ["Delay Cause","Percentage"]

delay_cause_percentage = delay_cause_percentage.round(2)

delay_cause_percentage.to_csv('../data/joint_dataset_delay_causes_dashboard.csv', index=False)

In [13]:
# Top airports by average delay

airport_avg_delay = completed_flights.groupby("AIRPORT_ORIGIN")["ARRIVAL_DELAY"].mean()

airport_avg_delay = airport_avg_delay.sort_values(ascending=False).head(10)

airport_avg_delay = airport_avg_delay.reset_index()

airport_avg_delay.columns = ["Airport","Average Arrival Delay"]

airport_avg_delay = airport_avg_delay.round(2)

airport_avg_delay.to_csv('../data/joint_dataset_airport_avg_delay_dashboard.csv', index=False)

In [14]:
# Airports with highest severe delay rate

completed_flights["SEVERE_DELAY"] = (completed_flights["ARRIVAL_DELAY"] > 60).astype(int)

airport_severe_delay = completed_flights.groupby("AIRPORT_ORIGIN")["SEVERE_DELAY"].mean() * 100

airport_severe_delay = airport_severe_delay.sort_values(ascending=False).head(10)

airport_severe_delay = airport_severe_delay.reset_index()

airport_severe_delay.columns = ["Airport","Severe Delay Rate (%)"]

airport_severe_delay = airport_severe_delay.round(2)

airport_severe_delay.to_csv('../data/joint_dataset_airport_severe_delay_dashboard.csv', index=False)